# Who's Riley's Crush? — Google Colab runner

Serves the single static page in this repo and exposes it publicly with [localtunnel](https://github.com/localtunnel/localtunnel).

**Run the cells top to bottom.** The last cell prints a public `https://*.loca.lt` URL — that's the live page. It's safe to re-run that last cell any time (e.g. after pulling an update) — it automatically stops the previous run first.

## 1. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/DeQuackDealer/confess-web.git"
BRANCH = "claude/build-rileyscrush-f0276h"  # switch to "main" once this branch is merged
REPO_DIR = "/content/confess-web"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print("Repo already present, pulling latest...")
    !cd {REPO_DIR} && git pull

## 2. Install Node.js and localtunnel

The page itself is static (no Node needed to serve it — that's done with Python's built-in `http.server` below), but `localtunnel` is an npm package, so Node is needed for that.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!npm install -g localtunnel
!node -v && npm -v

## 3. Serve the page and open a public tunnel

Starts a static file server over the repo (so `index.html` is served at `/`) and localtunnel in the background, waits for both to report ready, then prints the public URL.

In [ ]:
import subprocess, time, urllib.request

PORT = 8000
REPO_DIR = "/content/confess-web"
SERVER_LOG = "/content/server.log"
TUNNEL_LOG = "/content/tunnel.log"


def stop_previous_run():
    """Kills anything left running from an earlier run of this cell, so
    re-running it doesn't fail with an address-already-in-use error. Matches
    by command line via pkill rather than tracking PIDs in Python variables,
    so this works even across a kernel restart, not just within one session."""
    subprocess.run(["pkill", "-f", f"http.server {PORT}"], stderr=subprocess.DEVNULL)
    subprocess.run(["pkill", "-f", f"lt --port {PORT}"], stderr=subprocess.DEVNULL)
    time.sleep(1)


stop_previous_run()

print("Starting static file server...")
with open(SERVER_LOG, "w") as f:
    subprocess.Popen(
        ["python3", "-u", "-m", "http.server", str(PORT), "--directory", REPO_DIR],
        stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
    )

# Poll real HTTP readiness instead of trying to capture/parse the child
# process's printed output live — piping and streaming another process's
# stdout through Python threads has turned out to be unreliable in some
# notebook environments (buffering, output not surfacing in the cell, etc.).
# Reading server.log directly (below) is always available if you want to
# see it, but readiness itself is confirmed the only way that can't lie:
# actually making a request and getting a response.
server_ready = False
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
        server_ready = True
        break
    except Exception:
        time.sleep(1)

if not server_ready:
    print("\n⚠️ Server did not respond in time. Contents of server.log:")
    print(open(SERVER_LOG).read() or "(empty)")
else:
    print("Server is up.\n\nStarting localtunnel...")
    with open(TUNNEL_LOG, "w") as f:
        subprocess.Popen(
            ["lt", "--port", str(PORT)],
            stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
        )

    tunnel_url_line = None
    for _ in range(30):
        time.sleep(1)
        content = open(TUNNEL_LOG).read()
        if "your url is" in content:
            tunnel_url_line = content.strip()
            break

    if tunnel_url_line:
        print(f"\n✅ {tunnel_url_line}")
        print("   First-time visitors may see localtunnel's interstitial page — click 'Click to Continue'.")
    else:
        print("\n⚠️ localtunnel did not report a URL in time. Contents of tunnel.log:")
        print(open(TUNNEL_LOG).read() or "(empty)")

## 4. (Optional) Stop everything

Run this cell to shut down the server and tunnel without immediately starting a new run (cell 3 already does this automatically at the start of every run).

In [ ]:
stop_previous_run()